In [11]:
from mido import MidiFile, tempo2bpm
import networkx as nx
import os
import glob
import multiprocessing
import sqlite3
import json
import os
import glob
import concurrent.futures
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from sklearn.metrics import precision_recall_curve, auc
import random
import warnings
from itertools import combinations
from concurrent.futures import ThreadPoolExecutor, as_completed
import random
from datetime import datetime

def extract_midi_data(path):
    midi = MidiFile(path)

    # Extract metadata from the file path
    artist = os.path.basename(os.path.dirname(path)) if os.path.dirname(path) else 'Unknown'
    title = os.path.splitext(os.path.basename(path))[0]

    # Collect all events with absolute times
    all_events = []
    for track in midi.tracks:
        absolute_time = 0
        for msg in track:
            absolute_time += msg.time
            all_events.append({
                'time': absolute_time,
                'msg': msg,
                'track': track.name if hasattr(track, 'name') else 'unknown'
            })

    # Sort events by time
    all_events.sort(key=lambda e: e['time'])

    notes = []
    tempos = []
    time_signatures = []

    # Current state
    current_tempo = 500000  # default 120 BPM
    current_time_sig = {'numerator': 4, 'denominator': 4}
    current_beats_per_measure = 4
    current_seconds = 0.0
    current_beats = 0.0
    last_tick_time = 0

    ticks_per_beat = midi.ticks_per_beat

    for event in all_events:
        tick_delta = event['time'] - last_tick_time
        seconds_delta = (tick_delta / ticks_per_beat) * (current_tempo / 1000000.0)
        current_seconds += seconds_delta
        beats_delta = (seconds_delta * tempo2bpm(current_tempo)) / 60.0
        current_beats += beats_delta

        msg = event['msg']

        if msg.is_meta:
            if msg.type == 'set_tempo':
                current_tempo = msg.tempo
                tempos.append({
                    'time': event['time'],
                    'seconds': current_seconds,
                    'beats': current_beats,
                    'tempo': msg.tempo,
                    'bpm': tempo2bpm(msg.tempo),
                })
            elif msg.type == 'time_signature':
                current_time_sig = {'numerator': msg.numerator, 'denominator': msg.denominator}
                current_beats_per_measure = msg.numerator
                time_signatures.append({
                    'time': event['time'],
                    'seconds': current_seconds,
                    'beats': current_beats,
                    'numerator': msg.numerator,
                    'denominator': msg.denominator,
                })
        else:
            if msg.type in ('note_on', 'note_off'):
                measure = int(current_beats // current_beats_per_measure) + 1
                beat_in_measure = int(current_beats % current_beats_per_measure) + 1
                notes.append({
                    'time': event['time'],
                    'seconds': current_seconds,
                    'beats': current_beats,
                    'measure': measure,
                    'beat': beat_in_measure,
                    'type': msg.type,
                    'note': msg.note,
                    'velocity': msg.velocity,
                    'channel': msg.channel,
                })

        last_tick_time = event['time']

    # Sort notes by time for sequence
    notes.sort(key=lambda n: n['time'])

    # Compute vector features
    vector_features = []
    last_note_on = None
    for note in notes:
        if note['type'] != 'note_on':
            continue
        weight = 1.0 if note['beat'] == 1 else 0.8 if note['beat'] == 3 else 0.5 if note['beat'] in (2, 4) else 0.25
        if last_note_on is None:
            interval = 'Start'
            contour = 'Start'
        else:
            interval = note['note'] - last_note_on['note']
            contour = 'Up' if interval > 0 else 'Down' if interval < 0 else 'Repeat'
        vector_features.append({
            'note': note['note'],
            'beat': note['beat'],
            'interval': interval,
            'contour': contour,
            'weight': weight
        })
        last_note_on = note

    # Graph: nodes and edges
    graph_nodes = [{'id': i, 'note': n['note'], 'beat': n['beat'], 'measure': n['measure']} for i, n in enumerate(notes) if n['type'] == 'note_on']
    graph_edges = []
    note_on_indices = [i for i, n in enumerate(notes) if n['type'] == 'note_on']
    for j, idx in enumerate(note_on_indices):
        current = notes[idx]
        # Sequential edges
        if j < len(note_on_indices) - 1:
            next_idx = note_on_indices[j+1]
            next_note = notes[next_idx]
            interval = next_note['note'] - current['note']
            contour = 'Up' if interval > 0 else 'Down' if interval < 0 else 'Repeat'
            graph_edges.append({
                'from': j,
                'to': j+1,
                'type': 'sequential',
                'interval': interval,
                'contour': contour
            })
        # Hierarchical edges: connect strong beats
        # Find next strong beat (beat 1 or 3)
        for k in range(j+1, len(note_on_indices)):
            target = notes[note_on_indices[k]]
            if target['beat'] in (1, 3):
                interval = target['note'] - current['note']
                contour = 'Up' if interval > 0 else 'Down' if interval < 0 else 'Repeat'
                graph_edges.append({
                    'from': j,
                    'to': k,
                    'type': 'hierarchical',
                    'interval': interval,
                    'contour': contour
                })
                break  # only to next strong

    return {
        'ticks_per_beat': ticks_per_beat,
        'length_seconds': midi.length,
        'artist': artist,
        'title': title,
        'notes': notes,
        'tempos': tempos,
        'time_signatures': time_signatures,
        'vector_features': vector_features,
        'graph_nodes': graph_nodes,
        'graph_edges': graph_edges,
    }



In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
def store_song_data(conn, file_path, data):
    artist = data.get('artist') if data.get('artist') else os.path.basename(os.path.dirname(file_path)) if os.path.dirname(file_path) else 'Unknown'
    title = data.get('title') if data.get('title') else os.path.splitext(os.path.basename(file_path))[0]

    c = conn.cursor()
    c.execute('INSERT OR REPLACE INTO songs (file_path, artist, title, ticks_per_beat, length_seconds, tempos, time_signatures) VALUES (?, ?, ?, ?, ?, ?, ?)',
              (file_path, artist, title, data['ticks_per_beat'], data['length_seconds'], json.dumps(data['tempos']), json.dumps(data['time_signatures'])))
    song_id = c.lastrowid
    c.execute('INSERT OR REPLACE INTO vector_features (song_id, features) VALUES (?, ?)',
              (song_id, json.dumps(data['vector_features'])))
    c.execute('INSERT OR REPLACE INTO graph_data (song_id, nodes, edges) VALUES (?, ?, ?)',
              (song_id, json.dumps(data['graph_nodes']), json.dumps(data['graph_edges'])))
    # Removed conn.commit() to allow batch commits
    return song_id

In [ ]:


def create_db(db_path='midi_project.db'):
    conn = sqlite3.connect(db_path)

    # Optimizations for fast SQLite writing
    conn.execute("PRAGMA synchronous = OFF")
    conn.execute("PRAGMA journal_mode = MEMORY")
    conn.execute("PRAGMA temp_store = MEMORY")

    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS songs (
        id INTEGER PRIMARY KEY,
        file_path TEXT UNIQUE,
        artist TEXT,
        title TEXT,
        ticks_per_beat INTEGER,
        length_seconds REAL,
        tempos TEXT,
        time_signatures TEXT
    )''')
    c.execute('''CREATE TABLE IF NOT EXISTS vector_features (
        song_id INTEGER,
        features TEXT,
        FOREIGN KEY(song_id) REFERENCES songs(id)
    )''')
    c.execute('''CREATE TABLE IF NOT EXISTS graph_data (
        song_id INTEGER,
        nodes TEXT,
        edges TEXT,
        FOREIGN KEY(song_id) REFERENCES songs(id)
    )''')
    c.execute('''CREATE TABLE IF NOT EXISTS similarities (
        song_id INTEGER,
        song2_id INTEGER,
        vector_sim REAL,
        graph_sim REAL,
        FOREIGN KEY(song_id) REFERENCES songs(id),
        FOREIGN KEY(song2_id) REFERENCES songs(id)
    )''')
    conn.commit()
    return conn

def store_song_data(conn, file_path, data):
    artist = data.get('artist') if data.get('artist') else os.path.basename(os.path.dirname(file_path)) if os.path.dirname(file_path) else 'Unknown'
    title = data.get('title') if data.get('title') else os.path.splitext(os.path.basename(file_path))[0]

    c = conn.cursor()
    c.execute('INSERT OR REPLACE INTO songs (file_path, artist, title, ticks_per_beat, length_seconds, tempos, time_signatures) VALUES (?, ?, ?, ?, ?, ?, ?)',
              (file_path, artist, title, data['ticks_per_beat'], data['length_seconds'], json.dumps(data['tempos']), json.dumps(data['time_signatures'])))
    song_id = c.lastrowid
    c.execute('INSERT OR REPLACE INTO vector_features (song_id, features) VALUES (?, ?)',
              (song_id, json.dumps(data['vector_features'])))
    c.execute('INSERT OR REPLACE INTO graph_data (song_id, nodes, edges) VALUES (?, ?, ?)',
              (song_id, json.dumps(data['graph_nodes']), json.dumps(data['graph_edges'])))
    return song_id

def safe_extract(path):
    try:
        return extract_midi_data(path)
    except Exception as e:
        return None

base_path = '/content/drive/MyDrive/clean_midi'
db_path = '/content/drive/MyDrive/midi_project.db'

conn = create_db(db_path)

try:
    if not os.path.isdir(base_path):
        print(f"Directory {base_path} not found.")
        midi_files = []
    else:
        print(f"Scanning directory {base_path} for MIDI files.")
        midi_files = []
        for root, dirs, files in os.walk(base_path):
            for file in files:
                if file.lower().endswith('.mid') or file.lower().endswith('.midi'):
                    midi_files.append(os.path.join(root, file))

            if len(midi_files) % 1000 == 0 and len(midi_files) > 0:
                print(f"Found {len(midi_files)} files so far")

    print(f"\nTotal found: {len(midi_files)} MIDI files")

    if len(midi_files) > 0:
        batch_size = 200
        total_processed = 0

        with concurrent.futures.ProcessPoolExecutor() as executor:
            for i in range(0, len(midi_files), batch_size):
                batch = midi_files[i:i+batch_size]
                print(f"Processing batch {i//batch_size + 1} ({len(batch)} files)")

                results = list(executor.map(safe_extract, batch))

                conn.execute('BEGIN TRANSACTION')
                for j, data in enumerate(results):
                    if data is None:
                        continue
                    file_path = batch[j]
                    stored_name = os.path.basename(file_path)
                    data['artist'] = data.get('artist', 'Unknown')
                    data['title'] = data.get('title', os.path.splitext(stored_name)[0])
                    store_song_data(conn, stored_name, data)
                    total_processed += 1
                    if total_processed % 10 == 0:
                        print(f"Processed {total_processed}/{len(midi_files)} files")

                conn.commit()
                print(f"Processed {total_processed}/{len(midi_files)} files")

        print("Database populated")
finally:
    conn.close()

Scanning directory /content/drive/MyDrive/clean_midi for MIDI files.
Found 11000 files so far...

Total found: 17207 MIDI files
Processing batch 1 (200 files)
Processed 10/17207 files
Processed 20/17207 files
Processed 30/17207 files
Processed 40/17207 files
Processed 50/17207 files
Processed 60/17207 files
Processed 70/17207 files
Processed 80/17207 files
Processed 90/17207 files
Processed 100/17207 files
Processed 110/17207 files
Processed 120/17207 files
Processed 130/17207 files
Processed 140/17207 files
Processed 150/17207 files
Processed 160/17207 files
Processed 170/17207 files
Processed 180/17207 files
Processed 190/17207 files
Processed 197/17207 files
Processing batch 2 (200 files)
Processed 200/17207 files
Processed 210/17207 files
Processed 220/17207 files
Processed 230/17207 files
Processed 240/17207 files
Processed 250/17207 files
Processed 260/17207 files
Processed 270/17207 files
Processed 280/17207 files
Processed 290/17207 files
Processed 300/17207 files
Processed 310

In [5]:
conn = sqlite3.connect('/content/drive/MyDrive/midi_project.db')
c = conn.cursor()

c.execute('SELECT * FROM songs LIMIT 100')
rows = c.fetchall()
print("Songs table:")
for row in rows:
    print(row)

conn.close()

Songs table:
(1, 'Go Ahead Baby.mid', 'Jon Cleary', 'Go Ahead Baby', 384, 215.13346354166669, '[{"time": 0, "seconds": 0.0, "beats": 0.0, "tempo": 625000, "bpm": 96.0}]', '[{"time": 0, "seconds": 0.0, "beats": 0.0, "numerator": 4, "denominator": 4}]')
(2, 'Budweiser Polka.mid', 'Last', 'Budweiser Polka', 384, 165.79697166666665, '[{"time": 0, "seconds": 0.0, "beats": 0.0, "tempo": 517240, "bpm": 116.00030933415823}, {"time": 0, "seconds": 0.0, "beats": 0.0, "tempo": 517240, "bpm": 116.00030933415823}]', '[{"time": 0, "seconds": 0.0, "beats": 0.0, "numerator": 4, "denominator": 4}, {"time": 0, "seconds": 0.0, "beats": 0.0, "numerator": 4, "denominator": 4}]')
(3, 'An der Donau steht Marika.mid', 'Last', 'An der Donau steht Marika', 384, 160.29932291666668, '[{"time": 0, "seconds": 0.0, "beats": 0.0, "tempo": 555550, "bpm": 108.00108001080011}, {"time": 0, "seconds": 0.0, "beats": 0.0, "tempo": 555550, "bpm": 108.00108001080011}]', '[{"time": 0, "seconds": 0.0, "beats": 0.0, "numerator":